# Compare adapt_decomp v1.0 and v1.1 across every branch, full FDSI benchmark

The v1.1 counterpart of `../fdsi_benchmark/05_comparison_sv_loss_pareto_roa.ipynb`: pure disk
read, no run switches. Every branch `03*`/`04*` produced, both versions, over all 100 recordings.

| branch | v1.0 search | v1.1 search |
|---|---|---|
| `sv_loss` | `tpe_sv_median` | `mtpe_sv_mean` |
| `pareto_min_sv` | `tpe_pareto` | `mtpe_pareto` (min-sv selection) |
| `pareto_knee` | `tpe_pareto` | `mtpe_pareto` (knee selection) |
| `roa` | `tpe_roa` | `mtpe_roa` |

What changed between the versions: the search itself (multivariate TPE, `centroid_momentum`
searched, per-unit-mean `sv_loss`, CoV-ISI unit selection, worker processes), and the
application (`process_from_calib_end` instead of re-adapting from sample 0).

## Config

In [ ]:
# No run switches -- this notebook only reads the cache.

import yaml
from pathlib import Path

with open('../../configs/data_configs/fdsi_benchmark_grid.yaml') as f:
    grid = yaml.safe_load(f)

DATA_DIR = Path(grid['data_root'])           # Raw archive root: <sub>/{clean,noisy}/
OUTPUTS_ROOT = Path(grid['outputs_root'])    # Precomputed-outputs archive root
CAL_DIR = OUTPUTS_ROOT / 'calibration'       # Shared with v1.0 -- written by ../fdsi_benchmark/01_calibration
ADAPT_DIR = OUTPUTS_ROOT / 'adaptation_v1_1' # v1.1 results
ADAPT_DIR_V10 = OUTPUTS_ROOT / 'adaptation'  # Cached v1.0 results, read-only here
ADAPT_DIRS = {'v1.0': ADAPT_DIR_V10, 'v1.1': ADAPT_DIR}
OPT_DIR = ADAPT_DIR / 'optimisation'

FS, EXT_FACT = grid['fs'], grid['ext_fact']
TOL_SPIKE_MS, ROA_CAL_TH = grid['tol_spike_ms'], grid['roa_cal_th']
CAL_END, ISO_DUR = int(grid['cal_duration_s'] * FS), int(grid['iso_duration_s'] * FS)

SUBJECTS, CONDITIONS, SNR_LEVELS = grid['subjects'], grid['conditions'], grid['snr_levels']
TRIANGULAR = [c for c in CONDITIONS if 'triangular' in c]

POOL_CONDITIONS = grid['pool_conditions']
HOLDOUT_CONDITIONS = [c for c in CONDITIONS if c not in POOL_CONDITIONS]
OPTIM_SUB, OPTIM_SNR, N_TRIALS = grid['optim_sub'], grid['optim_snr'], grid['n_trials']

BASE_CONFIG_PATH = Path('../../configs/adapt_configs/default_muniverse.yaml')  # v1.0's base config
LR_MODE = 'fixed'   # v1.1 searches lr_fixed only -- the branch v1.0 promoted as its headline
DEVICE = 'cpu'      # deterministic, and MPS lacks the QR the sv update needs

BRANCHES = ['sv_loss', 'pareto_min_sv', 'pareto_knee', 'roa']

print(f'Apply-to-all grid: {len(SUBJECTS)} subjects x {len(CONDITIONS)} conditions x {len(SNR_LEVELS)} SNR levels '
      f'= {len(SUBJECTS) * len(CONDITIONS) * len(SNR_LEVELS)} recordings')
print(f'Calibration window [0, {CAL_END}) samples; adaptation starts at sample {CAL_END}')

## Aggregate RoA from disk

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '../fdsi_benchmark')  # for fdsi_common
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from adapt_decomp.utils.plots import plot_metric_heatmap, plot_metric_boxplot
import fdsi_common as fc
import fdsi_v11 as fv

sns.set_theme(style='whitegrid', font_scale=1.1)

CONFIG_LABELS = fv.configs(*BRANCHES)
CONFIG_ORDER = [label for label, _, _, _ in CONFIG_LABELS]

df_roa = fv.roa_table(ADAPT_DIRS, CAL_DIR, DATA_DIR, SUBJECTS, CONDITIONS, SNR_LEVELS,
                      CONFIG_LABELS, FS, TOL_SPIKE_MS)
df_post = fv.window_roa_table(ADAPT_DIRS, CAL_DIR, DATA_DIR, SUBJECTS, CONDITIONS, SNR_LEVELS,
                              CONFIG_LABELS, slice(CAL_END, None), FS, TOL_SPIKE_MS)

configs_present = [c for c in CONFIG_ORDER if len(df_roa) and c in set(df_roa['config'])]
print(f'{len(configs_present)}/{len(CONFIG_ORDER)} configs cached: {configs_present}')

## Hyperparameters -- promoted v1.0 vs v1.1 winners

In [ ]:
fv.hyperparameter_table(BRANCHES).round(5)

## Overall summary

In [ ]:
if len(df_roa):
    summary = (df_roa[df_roa['config'].isin(configs_present)]
               .groupby(['version', 'config'])['roa_pct']
               .agg(n_units='size', mean='mean', median='median',
                    pct_ge_90=lambda s: (s >= 90).mean() * 100)
               .round(2))
    display(summary)

## Paired per-unit comparison, v1.0 -> v1.1

In [ ]:
if len(df_roa) and len(df_post):
    display(pd.concat({'full recording': fv.paired_summary(df_roa, fv.pairs(*BRANCHES)),
                       'after calibration': fv.paired_summary(df_post, fv.pairs(*BRANCHES))}).round(3))

## By condition and by SNR

In [ ]:
if len(df_roa):
    fig, axs = plt.subplots(1, 2, figsize=(20, 5), layout='constrained')
    plot_metric_heatmap(df_roa, 'roa_pct', configs_present, CONDITIONS, SNR_LEVELS,
                        agg='mean', axs=[axs[0]])
    plot_metric_heatmap(df_roa, 'roa_pct', configs_present, CONDITIONS, SNR_LEVELS,
                        agg='pct_ge_threshold', threshold=90, axs=[axs[1]])
    plt.show()

## Per-unit RoA distribution, one panel per version

In [ ]:
if len(df_roa):
    fig, axs = plt.subplots(1, 1, figsize=(16, 5.5), layout='constrained')
    plot_metric_boxplot(df_roa, 'roa_pct', configs_present, CONDITIONS, SNR_LEVELS, axs=[axs])
    plt.show()

## Search cost -- trial wall time

In [ ]:
import pickle

rows = []
for search, search_dir in fv.SEARCH_DIRS.items():
    for version, root in ADAPT_DIRS.items():
        v10_dir = {'sv_loss': 'tpe_sv_median', 'pareto': 'tpe_pareto', 'roa': 'tpe_roa'}[search]
        path = (root / 'optimisation' / fc.LR_MODE_TOKEN[LR_MODE]
                / (search_dir if version == 'v1.1' else v10_dir) / 'study.pkl')
        if not path.exists():
            continue
        with open(path, 'rb') as f:
            study = pickle.load(f)
        seconds = fv.trial_durations(study)
        rows.append({'search': search, 'version': version, 'n_trials': len(seconds),
                     'median_trial_s': seconds.median(),
                     'total_s': study.user_attrs.get('wall_time_s', seconds.sum())})

if rows:
    display(pd.DataFrame(rows).set_index(['search', 'version']).round(1))
else:
    print('No cached studies found.')

### Reading this

Read the **after calibration** block for the verdict. Inside `[0, CAL_END)` v1.1 emits CBSS's
own calibration spikes while v1.0 re-adapted over that window, so a full-recording difference
mixes that bookkeeping change in with the adaptation change; `[CAL_END, end)` is the window both
versions genuinely adapted.

The pairing is per unit, and legitimate because both versions start from the *same* cached
calibration, so a branch's v1.0 and v1.1 rows describe the same motor units in the same order.

`sv_loss` scalars are not comparable across versions -- v1.0 reduced the per-batch unit sum,
v1.1 the per-unit mean -- so compare branches through RoA, not through their objective values.